In [ ]:
# -*- coding: utf-8 -*-
# page.encoding='utf-8'

from catmap import ReactionModel

mkm_file = 'alkene_epoxidation.mkm'
model = ReactionModel(setup_file=mkm_file)

# Do not request the default 'rate'/'coverage' maps twice.
for variable in ['production_rate','rate','rate_control','coverage','selectivity_control','rxn_order']:
    if variable not in model.output_variables:
        model.output_variables.append(variable)

# CatMAP serializes results to a Python-readable .log and binary .pkl.
# NumPy scalars in transition_state_scaling_coefficients can break log reload
# (NameError: np is not defined); register NumPy in the log namespace.
model._log_imports += '\nimport numpy as np\n'

# Use CatMAP's native binary storage for non-literal matrix / mpf objects.
# Prevents "header_evaluation: fail - could not save ..." on reloading.
for attr in [
    'adsorbate_coefficient_matrix',
    'coefficient_matrix',
    'kBT',
    'total_coefficient_dict',
    'transition_state_coefficient_matrix',
    'transition_state_scaling_matrix',
    'transition_state_scaling_coefficients',
]:
    if attr not in model._pickle_attrs:
        model._pickle_attrs.append(attr)

model.run()

from catmap import analyze
vm = analyze.VectorMap(model)
vm.plot_variable = 'rate' #tell the model which output to plot
vm.log_scale = True #rates should be plotted on a log-scale
vm.min = 1e-25 #minimum rate to plot
vm.max = 1e3 #maximum rate to plot
vm.plot(save='rate.pdf') #draw the plot and save it as "rate.pdf"

vm.unique_only = False
vm.plot(save='all_rates.pdf')
vm.unique_only = True

vm.production_rate_map = model.production_rate_map #attach map
vm.threshold = 1e-30 #do not plot rates below this
vm.plot_variable = 'production_rate'
vm.plot(save='production_rate.pdf')

vm.descriptor_labels = ['OOH formation energy [eV]', 'O formation energy [eV]']
vm.subplots_adjust_kwargs = {'left':0.2,'right':0.8,'bottom':0.15}
vm.plot(save='pretty_production_rate.pdf')

vm.plot_variable = 'coverage'
vm.log_scale = False
vm.min = 0
vm.max = 1
vm.plot(save='coverage.pdf')

vm.include_labels = ['OOH_s', 'O_s']
vm.plot(save='OOH_O_coverage.pdf')

sa = analyze.ScalingAnalysis(model)
sa.plot(save='scaling.pdf')



In [ ]:
from pathlib import Path
from catmap.model import ReactionModel

output_variable = 'production_rate'
logfile = Path('alkene_epoxidation.log')
if not logfile.is_file():
    raise FileNotFoundError(f'Model log not found: {logfile}. Run Cell 1 first.')
model = ReactionModel(setup_file=str(logfile))
if not hasattr(model, output_variable + '_map'):
    raise ValueError(f'{output_variable} was not computed. Add it to Cell 1 output_variables and rerun.')

if output_variable == 'rate_control':
    dim = 2
else:
    dim = 1

labels = model.output_labels[output_variable]

def flatten_2d(output):
    "Helper function for flattening rate_control output"
    flat = []
    for x in output:
        flat+= x
    return flat

#flatten rate_control labels
if output_variable == 'rate_control':
    flat_labels = []
    for i in labels[0]:
        for j in labels[1]:
            flat_labels.append('d'+i+'/d'+j)
    labels = flat_labels

#flatten elementary-step specific labels
if output_variable in ['rate','rate_constant','forward_rate_constant','reverse_rate_constant']:
    str_labels = []
    for label in labels:
        states = ['+'.join(s) for s in label]
        if len(states) == 2:
            new_label = '<->'.join(states)
        else:
            new_label = states[0]+'<->'+states[1]+'->'+states[2]
        str_labels.append(new_label)
    labels = str_labels

table = '\t'.join(list(['descriptor-'+d for d in model.descriptor_names])+list(labels))+'\n'

for pt, output in getattr(model,output_variable+'_map'):
    if dim == 2:
        output = flatten_2d(output)
    table += '\t'.join([str(float(i)) for i in pt+output])+'\n'

f = open(output_variable+'_table.txt','w')
f.write(table)
f.close()

In [ ]:
from catmap.model import ReactionModel

model = ReactionModel(setup_file='alkene_epoxidation.log')

#for descriptors, cvgs in model.coverage_map:
   # print( 'descriptors:', descriptors)
   #print( 'coverages', cvgs)
    
labels = model.output_labels['coverage']
for descriptors ,cvg in model.coverage_map:
    print( 'descriptors',descriptors)
    print( 'intermediates',labels)
    print( 'coverages', [float(c) for c in cvg])